# Trích đặc trưng từ EfficientNet-B0 đã tinh chỉnh

**BTL Khai phá dữ liệu – HK261.**

`extract-features` dùng EfficientNet-B0 học trước trên ImageNet và giữ đóng băng. Ở CNN-LSTM giai đoạn B, EfficientNet được tinh chỉnh cùng phần BiLSTM trên video ký hiệu (Macro F1 private 0,687 so với 0,415 khi đóng băng). Notebook này dùng EfficientNet **đã tinh chỉnh** đó làm bộ trích đặc trưng thứ hai, để so sánh các thuật toán phân loại trên hai bộ đặc trưng.

| File đầu ra | Kích thước | Nội dung |
|---|---|---|
| `feat_ft_train.npy` / `feat_ft_public.npy` / `feat_ft_private.npy` | (N, 16, 1280) | vector 1.280 chiều của từng khung, cùng định dạng với `feat_*.npy` |
| `emb_ft_train.npy` / `emb_ft_public.npy` / `emb_ft_private.npy` | (N, 512) | vector của cả video sau BiLSTM + attention |
| `att_ft_train.npy` / `att_ft_public.npy` / `att_ft_private.npy` | (N, 16) | trọng số attention của từng khung |

Video đi qua đúng đường xử lý của `preprocess.py` (bỏ khung đơn sắc, lấy đều 16 khung, chuẩn hoá ImageNet), không tăng cường. Thứ tự dòng khớp `manifest_clean.csv` và `test_manifest.csv`.

**Lưu ý khi dùng:** CNN đã được tinh chỉnh trên fold 1–4, nên đặc trưng của các video này đã "thấy" nhãn của chúng; cross-validation trên fold 1–4 với bộ đặc trưng này sẽ lạc quan. Fold 0 (chỉ dùng để chọn epoch dừng) và tập private vẫn là đánh giá khách quan.

In [ ]:
import os, sys, json, time, shutil, warnings
from pathlib import Path

warnings.filterwarnings('ignore', category=SyntaxWarning)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0
from sklearn.metrics import accuracy_score, f1_score

T = 16
N_CLASSES = 100
VAL_FOLD = 0
BATCH_VIDEOS = int(os.environ.get('BATCH_VIDEOS', '8'))
NUM_WORKERS = int(os.environ.get('NUM_WORKERS', 0 if sys.platform == 'darwin' else min(4, os.cpu_count() or 1)))
LIMIT = int(os.environ.get('LIMIT', '0'))

ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    ROOT = Path('/kaggle')
    SEARCH_ROOTS, OUT = [ROOT / 'input'], ROOT / 'working'
else:
    ROOT = Path(os.environ.get('PROJECT_ROOT', Path.cwd().parent))
    SEARCH_ROOTS = [ROOT / 'models' / 'features', ROOT / 'models' / 'cnn_lstm' / 'output', ROOT / 'data', ROOT / 'processing']
    OUT = ROOT / 'models' / 'features_ft'
OUT = Path(os.environ.get('OUT_DIR', OUT))
OUT.mkdir(parents=True, exist_ok=True)


def find_file(name):
    for root in SEARCH_ROOTS:
        hits = sorted(root.rglob(name)) if root.exists() else []
        if hits:
            return hits[0]
    raise FileNotFoundError(f'Không tìm thấy {name} trong {SEARCH_ROOTS}')


def rel(path):
    return path.relative_to(ROOT) if path.is_relative_to(ROOT) else path


FEAT_DIR = find_file('label_names.json').parent
CKPT = find_file('cnn_lstm_B.pth')
B_PRIVATE = find_file('cnn_lstm_B_private.csv')
DATASET = find_file('label_mapping.pkl').parent
PP_DIR = FEAT_DIR if (FEAT_DIR / 'preprocess.py').exists() else find_file('preprocess.py').parent
sys.path.insert(0, str(PP_DIR))
import preprocess as pp

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'

_default_unraisable = sys.unraisablehook

def _hide_loader_shutdown(info):
    if '_MultiProcessingDataLoaderIter' not in getattr(info.object, '__qualname__', ''):
        _default_unraisable(info)


sys.unraisablehook = _hide_loader_shutdown

print('Manifest, nhãn:', rel(FEAT_DIR))
print('Trọng số      :', rel(CKPT))
print('Video         :', rel(DATASET))
print('Output        :', rel(OUT))
print('Thiết bị      :', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '')

Manifest, nhãn: models/features
Trọng số      : models/cnn_lstm/output/cnn_lstm_B.pth
Video         : data/dataset
Output        : models/features_ft
Thiết bị      : mps 


## 1. Nạp mô hình CNN-LSTM giai đoạn B

Kiến trúc giống hệt notebook `cnn_lstm`: EfficientNet-B0 (bỏ lớp phân loại) → chuẩn hoá đặc trưng theo trung bình và độ lệch chuẩn đã lưu → chiếu 512 chiều → BiLSTM 2 lớp → attention → lớp phân loại. Ở đây mô hình chỉ chạy suy luận (`eval`, không tính gradient).

In [2]:
class TemporalHead(nn.Module):
    def __init__(self, in_dim, proj=512, hidden=256, layers=2, drop=0.3, drop_out=0.4):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, proj), nn.LayerNorm(proj), nn.ReLU(), nn.Dropout(drop))
        self.lstm = nn.LSTM(proj, hidden, num_layers=layers, batch_first=True, bidirectional=True,
                            dropout=drop if layers > 1 else 0.0)
        self.att = nn.Sequential(nn.Linear(2 * hidden, 128), nn.Tanh(), nn.Linear(128, 1, bias=False))
        self.cls = nn.Sequential(nn.Dropout(drop_out), nn.Linear(2 * hidden, N_CLASSES))

    def embed(self, x):
        h, _ = self.lstm(self.proj(x))
        alpha = torch.softmax(self.att(h).squeeze(-1).float(), dim=1)
        return (alpha.unsqueeze(-1) * h.float()).sum(1), alpha


class CNNLSTM(nn.Module):
    def __init__(self, feat_mean, feat_std):
        super().__init__()
        self.cnn = efficientnet_b0(weights=None)
        self.cnn.classifier = nn.Identity()
        self.head = TemporalHead(1280)
        self.register_buffer('mean', torch.as_tensor(feat_mean, dtype=torch.float32))
        self.register_buffer('std', torch.as_tensor(feat_std, dtype=torch.float32))

    def forward(self, x):
        b = x.shape[0]
        f = self.cnn(x.flatten(0, 1).contiguous(memory_format=torch.channels_last)).view(b, T, -1)
        z, alpha = self.head.embed((f.float() - self.mean) / self.std)
        return f, z, alpha, self.head.cls(z)


ckpt = torch.load(CKPT, map_location='cpu', weights_only=False)
model = CNNLSTM(ckpt['feat_mean'], ckpt['feat_std'])
missing, unexpected = model.load_state_dict(ckpt['state_dict'], strict=False)
assert not unexpected and set(missing) <= {'mean', 'std'}, (missing, unexpected)
model = model.eval().to(DEVICE).to(memory_format=torch.channels_last)
label_names = json.load(open(FEAT_DIR / 'label_names.json', encoding='utf-8'))
assert ckpt['label_names'] == label_names
print(f"Checkpoint epoch {ckpt['epoch']} | {sum(p.numel() for p in model.parameters()) / 1e6:.2f} triệu tham số")

Checkpoint epoch 13 | 7.94 triệu tham số


## 2. Trích đặc trưng cho toàn bộ video

In [3]:
manifest = pd.read_csv(FEAT_DIR / 'manifest_clean.csv')
test_manifest = pd.read_csv(FEAT_DIR / 'test_manifest.csv')
public = test_manifest[test_manifest.split == 'public_test'].reset_index(drop=True)
private = test_manifest[test_manifest.split == 'private_test'].reset_index(drop=True)
train = manifest
if LIMIT:
    train, public, private = manifest.head(LIMIT), public.head(LIMIT), private.head(LIMIT)

dirs = pp.class_dirs(DATASET / 'train')
splits = {
    'train': ([DATASET / 'train' / dirs[pp.nfc(c)] / f for c, f in zip(train.cls, train.file)], train.valid_idx.tolist()),
    'public': ([DATASET / 'public_test' / f for f in public.file], None),
    'private': ([DATASET / 'private_test' / f for f in private.file], None),
}


class Clips(Dataset):
    def __init__(self, paths, valid):
        self.paths, self.valid = paths, valid

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        v = self.valid[i] if self.valid is not None else None
        return torch.from_numpy(pp.normalize(pp.load_clip(self.paths[i], valid_idx=v)))


@torch.no_grad()
def run(paths, valid):
    dl = DataLoader(Clips(paths, valid), batch_size=BATCH_VIDEOS, num_workers=NUM_WORKERS,
                    pin_memory=DEVICE.type == 'cuda')
    feats, embs, atts, probs = [], [], [], []
    for x in dl:
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
            f, z, alpha, logits = model(x.to(DEVICE, non_blocking=True))
        feats.append(f.float().cpu().numpy().astype(np.float16))
        embs.append(z.float().cpu().numpy())
        atts.append(alpha.float().cpu().numpy())
        probs.append(torch.softmax(logits.float(), -1).cpu().numpy())
    return [np.concatenate(a) for a in (feats, embs, atts, probs)]


out, timing = {}, {}
for name, (paths, valid) in splits.items():
    t0 = time.time()
    out[name] = run(paths, valid)
    timing[name] = round(time.time() - t0)
    print(f'{name:8s}: {len(paths)} video trong {timing[name]}s | đặc trưng {out[name][0].shape} | '
          f'vector video {out[name][1].shape}')

train   : 2772 video trong 198s | đặc trưng (2772, 16, 1280) | vector video (2772, 512)
public  : 1630 video trong 161s | đặc trưng (1630, 16, 1280) | vector video (1630, 512)
private : 2859 video trong 308s | đặc trưng (2859, 16, 1280) | vector video (2859, 512)


## 3. Kiểm tra tính đúng

Cho các đặc trưng vừa trích đi qua phần phân loại của chính mô hình B, kết quả phải khớp với notebook `cnn_lstm`: Macro F1 fold 0 khoảng 0,859 và dự đoán private trùng với file nộp `cnn_lstm_B_private.csv` (sai khác rất nhỏ có thể do làm tròn số thực 16 bit trên GPU).

In [4]:
for name, (f, z, a, _) in out.items():
    assert np.isfinite(f.astype(np.float32)).all() and np.isfinite(z).all()
    assert np.allclose(a.sum(1), 1, atol=1e-3)

if not LIMIT:
    y, fold = manifest.label_idx.values, manifest.fold.values
    pred = out['train'][3].argmax(1)
    va = fold == VAL_FOLD
    print(f'Fold 0: Macro F1 {f1_score(y[va], pred[va], average="macro"):.4f} | accuracy {accuracy_score(y[va], pred[va]):.4f}')
    print(f'Fold 1–4 (CNN đã học các video này): accuracy {accuracy_score(y[~va], pred[~va]):.4f}')
    b_sub = pd.read_csv(B_PRIVATE)
    assert (b_sub.video_name.values == private.file.values).all()
    agree = (np.array([label_names[i] for i in out['private'][3].argmax(1)]) == b_sub.label.values).mean()
    print(f'Dự đoán private trùng file nộp cnn_lstm_B: {agree:.2%}')

Fold 0: Macro F1 0.8589 | accuracy 0.9153
Fold 1–4 (CNN đã học các video này): accuracy 0.9986
Dự đoán private trùng file nộp cnn_lstm_B: 99.23%


## 4. Lưu kết quả

Chép kèm manifest và `label_names.json` để notebook so sánh thuật toán chỉ cần gắn output của notebook này và của `extract-features`.

In [5]:
for name, (f, z, a, _) in out.items():
    np.save(OUT / f'feat_ft_{name}.npy', f)
    np.save(OUT / f'emb_ft_{name}.npy', z.astype(np.float32))
    np.save(OUT / f'att_ft_{name}.npy', a.astype(np.float32))
for fname in ['manifest_clean.csv', 'test_manifest.csv', 'label_names.json']:
    shutil.copy(FEAT_DIR / fname, OUT / fname)

meta = {'source': 'cnn_lstm_B.pth', 'checkpoint_epoch': int(ckpt['epoch']), 'feat_dim': 1280, 'emb_dim': 512, 'T': T,
        'dtype_feat': 'float16', 'limit': LIMIT, 'n_train': len(train), 'n_public': len(public),
        'n_private': len(private), 'seconds': timing, 'device': str(DEVICE), 'torch': torch.__version__}
json.dump(meta, open(OUT / 'meta_ft.json', 'w'), indent=2)
for p in sorted(q for q in OUT.iterdir() if q.is_file() and not q.name.startswith(('.', '__'))):
    print(f'{p.name:22s} {p.stat().st_size / 1e6:8.1f} MB')

att_ft_private.npy          0.2 MB
att_ft_public.npy           0.1 MB
att_ft_train.npy            0.2 MB
emb_ft_private.npy          5.9 MB
emb_ft_public.npy           3.3 MB
emb_ft_train.npy            5.7 MB
feat_ft_private.npy       117.1 MB
feat_ft_public.npy         66.8 MB
feat_ft_train.npy         113.5 MB
label_names.json            0.0 MB
manifest_clean.csv          0.5 MB
meta_ft.json                0.0 MB
test_manifest.csv           0.3 MB
